# Step 3 — Multi-view RGB: **true Gaussian splat** (gsplat)

**Pipeline position:** rendering stage (after **`02_rendering_mesh_debug.ipynb`**, before **`04_vlm_features_debug.ipynb`**).

**Why does my image look like a sphere?** **gsplat draws whatever is in the `.ply`.** The default test file `tiny_gaussians.ply` is literally **Gaussians on a spherical shell** — the raster is supposed to look like a ball. For a **mug, chair, etc.**, point **`SPLAT_PLY`** at a **reconstruction** export (e.g. SAM3D’s **`outputs/reconstructions/<stem>/gaussian.ply`**) where the optimizer placed Gaussians on the real object. The renderer does not infer object shape from nothing.

**Output:** RGB frames go to **`exports/gaussian_splat/`** (not under `outputs/`, which is **`.gitignored`** and often **hidden in Cursor’s file tree**). Files: **`gsplat_rgb_000.png`**, …, and **`gsplat_rgb_000.jpg`** for view 0. Depth colormap: **`gsplat_depth_000.png`**.

**In-notebook:** the last cell embeds a preview of view 0.

**AffordSplat:** leave **`SPLAT_PLY`** unset and **`USE_TINY_EXAMPLE`** false — the config cell uses your **local mirror** (`AFFORDANCE_AFFORDSPLAT_ROOT`, `paths.affordsplat_root`, **`AFFORDANCE_DATA_ROOT`/data_root + `Seen/`**, **`/workspace/data`**, **`/data`**). With **`AFFORDSPLAT_RANDOM_SPLAT=True`** (default) it picks a **random** training row each time you run the config cell; set false for the first row only. **`GSPLAT_SUBSAMPLE_SEED`**: `None` = a new random Gaussian subset when the PLY exceeds `max_points`. Toggle **`AUTO_LOAD_AFFORDSPLAT`** or **`SPLAT_PLY`** to override. See **`01_affordsplat_dataloader.ipynb`**.

**PLY path:** set **`SPLAT_PLY`** in the configuration cell (or env `AFFORDANCE_SPLAT_PLY`, or yaml / auto-discovery as documented there).

**Requirements:** GPU + **`gsplat`** in the active kernel. If gsplat fails, a pyrender **centre preview** may still run and saves **`preview_rgb_*.png`**.

**Pass checklist:**
- [ ] After running the render cell, **`exports/gaussian_splat/`** appears in the repo sidebar (or use the printed **`EXPORT_DIR`** absolute path)
- [ ] **`gsplat_rgb_000.png`** matches **your** object once **`SPLAT_PLY`** points at your object `.ply` (not the tiny test sphere)

In [ ]:
%matplotlib inline


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# === Gaussian splat input (edit here) ===
# Repo-relative or absolute path to a 3DGS .ply export:
# SPLAT_PLY = "outputs/reconstructions/watering_can/gaussian.ply"
SPLAT_PLY: str | None = None

# Smoke test: force the bundled synthetic sphere (skip auto-discovery).
USE_TINY_EXAMPLE: bool = False

# Local AffordSplat mirror (e.g. /workspace/data/Seen/...): auto-pick first Gaussian .ply when nothing else is set.
AUTO_LOAD_AFFORDSPLAT: bool = True
AFFORDSPLAT_SUBSET: str = "Seen"
AFFORDSPLAT_SPLIT: str = "train"
# Random AffordSplat PLY each time this cell runs (and random gsplat subsample when N > max_points).
AFFORDSPLAT_RANDOM_SPLAT: bool = True
GSPLAT_SUBSAMPLE_SEED: int | None = None  # None = different Gaussian subset each render; set 0 to fix

_cwd = Path.cwd().resolve()
for _root in [_cwd, *_cwd.parents]:
    if (_root / "pyproject.toml").is_file() and (_root / "src").is_dir():
        sys.path.insert(0, str(_root / "src"))
        break
else:
    raise RuntimeError("Run this notebook from the repository (or a subfolder).")

from utils.config import load_config, project_root, resolve_path

from datasets.affordsplat_local_dataset import peek_first_affordsplat_row, sample_random_affordsplat_row

ROOT = project_root()
_cfg = load_config()


def resolve_splat_path() -> tuple[Path, str]:
    """env → SPLAT_PLY → yaml → AffordSplat (optional) → newest reconstruction → tiny example."""
    env = os.environ.get("AFFORDANCE_SPLAT_PLY", "").strip()
    if env:
        return resolve_path(env, root=ROOT), "AFFORDANCE_SPLAT_PLY (environment)"

    if SPLAT_PLY:
        return resolve_path(SPLAT_PLY, root=ROOT), "SPLAT_PLY (notebook cell)"

    yaml_splat = _cfg.get("rendering", {}).get("splat_path")
    if yaml_splat:
        p = resolve_path(yaml_splat, root=ROOT)
        if p.is_file():
            return p, "configs/default.yaml → rendering.splat_path"

    if AUTO_LOAD_AFFORDSPLAT and not USE_TINY_EXAMPLE:
        if AFFORDSPLAT_RANDOM_SPLAT:
            row = sample_random_affordsplat_row(
                cfg=_cfg, subset=AFFORDSPLAT_SUBSET, split=AFFORDSPLAT_SPLIT, seed=None
            )
        else:
            row = peek_first_affordsplat_row(
                cfg=_cfg, subset=AFFORDSPLAT_SUBSET, split=AFFORDSPLAT_SPLIT
            )
        if row is not None:
            tag = "auto: AffordSplat (random)" if AFFORDSPLAT_RANDOM_SPLAT else "auto: AffordSplat (first)"
            return row.splat_path.resolve(), f"{tag} ({row.sample_id})"

    if not USE_TINY_EXAMPLE:
        recon_root = ROOT / "outputs" / "reconstructions"
        if recon_root.is_dir():
            candidates = sorted(
                recon_root.glob("*/gaussian.ply"),
                key=lambda p: p.stat().st_mtime,
                reverse=True,
            )
            if candidates:
                rel = candidates[0].relative_to(ROOT)
                return candidates[0].resolve(), f"auto: newest {rel}"

    tiny = ROOT / "examples" / "gaussian_splat" / "tiny_gaussians.ply"
    return tiny.resolve(), "default: examples/gaussian_splat/tiny_gaussians.ply"


SPLAT_PATH, SPLAT_SOURCE = resolve_splat_path()
print("SPLAT_PATH:", SPLAT_PATH)
print("source:", SPLAT_SOURCE)
print("exists:", SPLAT_PATH.is_file())
if "tiny_gaussians" in SPLAT_PATH.name.lower():
    print(
        "\n>>> Using the bundled TEST .ply (Gaussians arranged on a sphere). "
        "gsplat is correct — the SCENE is a ball, not a mug/chair.\n"
        "    To see YOUR object: set SPLAT_PLY to your reconstruction export, e.g.\n"
        "        SPLAT_PLY = \\\"outputs/reconstructions/<object_stem>/gaussian.ply\\\"\\n"
        "    (run SAM3D / 01 notebook first, or copy any Inria-style object .ply here), "
        "then re-run this cell and the render cell.\n",
    )


In [ ]:
from pathlib import Path

import numpy as np
from PIL import Image as PILImage

from rendering.gaussian_point_renderer import render_gaussian_splat_views
from rendering.renderer import build_render_config

cfg = load_config()
mrc = build_render_config(cfg)
# Not under outputs/ — that path is .gitignored and often hidden in Cursor.
OUT_DIR = ROOT / "exports" / "gaussian_splat"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("rendering from:", SPLAT_PATH, f"({SPLAT_SOURCE})")
print("EXPORT_DIR (open this folder):", OUT_DIR.resolve())

# --- True 3DGS raster (CUDA + gsplat) ---
views_gsplat = None
gsplat_err: str | None = None
if SPLAT_PATH.is_file():
    try:
        from rendering.gaussian_gsplat_renderer import render_gaussian_splat_gsplat_views

        views_gsplat = render_gaussian_splat_gsplat_views(
            SPLAT_PATH, mrc, max_points=200_000, normalize_scene=True, seed=GSPLAT_SUBSAMPLE_SEED
        )
        print("gsplat views:", len(views_gsplat))
    except Exception as exc:
        gsplat_err = repr(exc)
        print("gsplat failed (need gsplat + CUDA GPU):", gsplat_err)
else:
    gsplat_err = f"missing PLY: {SPLAT_PATH}"
    print(gsplat_err)

# --- Fallback: centre preview (pyrender), same PLY ---
views_splat_preview = None
if views_gsplat is None and SPLAT_PATH.is_file():
    views_splat_preview = render_gaussian_splat_views(
        SPLAT_PATH, mrc, max_points=50_000, seed=0, preview_mode="auto"
    )
    print("fallback centre-preview views:", len(views_splat_preview), "(not full splatting)")


def save_rgb_stack(views: list, stem: str) -> list[Path]:
    """Write each view as PNG; view 0 also as JPEG for quick sharing."""
    paths: list[Path] = []
    for i, v in enumerate(views):
        png = OUT_DIR / f"{stem}_{i:03d}.png"
        PILImage.fromarray(v.rgb).save(png)
        paths.append(png)
    if views:
        jpg = OUT_DIR / f"{stem}_000.jpg"
        PILImage.fromarray(views[0].rgb).save(jpg, quality=92, optimize=True)
        paths.append(jpg)
    return paths


saved: list[Path] = []
if views_gsplat is not None:
    saved = save_rgb_stack(views_gsplat, "gsplat_rgb")
elif views_splat_preview is not None:
    saved = save_rgb_stack(views_splat_preview, "preview_rgb")

for p in saved:
    print("wrote:", p.relative_to(ROOT) if p.is_relative_to(ROOT) else p)



In [ ]:
from io import BytesIO

import numpy as np
from IPython.display import Image, Markdown, display
from matplotlib import cm
from PIL import Image as PILImage


def _png_bytes(rgb_u8: np.ndarray) -> bytes:
    buf = BytesIO()
    PILImage.fromarray(rgb_u8).save(buf, format="PNG")
    return buf.getvalue()


def _depth_preview_png(depth: np.ndarray) -> tuple[np.ndarray, float, float]:
    d = depth.astype(np.float64)
    mask = np.isfinite(d) & (d > 0)
    dmin = float(d[mask].min()) if mask.any() else 0.0
    dmax = float(d[mask].max()) if mask.any() else 1.0
    norm = np.zeros_like(d, dtype=np.float64)
    norm[mask] = (d[mask] - dmin) / max(dmax - dmin, 1e-8)
    norm = np.clip(norm, 0.0, 1.0)
    depth_rgb = (cm.magma(norm)[:, :, :3] * 255.0).astype(np.uint8)
    depth_rgb[~mask] = 0
    return depth_rgb, dmin, dmax


hero_png = OUT_DIR / ("gsplat_rgb_000.png" if views_gsplat is not None else "preview_rgb_000.png")
if hero_png.is_file():
    display(
        Markdown(
            f"### On disk (open in your IDE / file manager)\n"
            f"- **Hero RGB:** `{hero_png}`\n"
            f"- **Folder:** `{OUT_DIR}`\n"
        )
    )

if views_gsplat is not None:
    display(
        Markdown(
            "### True Gaussian splat — view 0 (**gsplat**)\n"
            "Ellipsoidal 3DGS raster (SH degree-0 only). Same pixels as **`gsplat_rgb_000.png`**."
        )
    )
    g = views_gsplat[0].rgb
    w = min(900, g.shape[1])
    display(Image(data=_png_bytes(g), format="png", embed=True, width=w))
    print("gsplat view 0 |", g.shape, "| mean", round(float(g.mean()), 2))
    depth_rgb, d0, d1 = _depth_preview_png(views_gsplat[0].depth)
    depth_png = OUT_DIR / "gsplat_depth_000.png"
    PILImage.fromarray(depth_rgb).save(depth_png)
    print("wrote depth preview:", depth_png.relative_to(ROOT))
    display(Markdown("### Depth (colormap) — saved next to RGB"))
    display(Image(data=_png_bytes(depth_rgb), format="png", embed=True, width=w))
    print("depth range:", d0, "…", d1)
else:
    display(
        Markdown(
            "### True Gaussian splat (gsplat) unavailable\n"
            f"```{gsplat_err}```\n"
            "Need **CUDA** + **`gsplat`** in this kernel. If preview ran, see **`exports/gaussian_splat/preview_rgb_*.png`**."
        )
    )
    if views_splat_preview is not None:
        display(
            Markdown(
                "### Fallback: centre preview only (not full splatting)\n"
                "Files: **`preview_rgb_*.png`**, **`preview_rgb_000.jpg`**"
            )
        )
        p0 = views_splat_preview[0].rgb
        display(Image(data=_png_bytes(p0), format="png", embed=True, width=min(900, p0.shape[1])))



## CLI batch export

**True splat (GPU):**

    PYTHONPATH=src python scripts/render_gaussian_views.py --backend gsplat \
      --splat_path examples/gaussian_splat/tiny_gaussians.ply \
      --output_dir outputs/gaussian_renders/demo

**Centre preview only (pyrender / EGL, no gsplat):**

    PYTHONPATH=src python scripts/render_gaussian_views.py --backend preview \
      --splat_path examples/gaussian_splat/tiny_gaussians.ply \
      --output_dir outputs/gaussian_renders/demo

Replace `tiny_gaussians.ply` with your reconstruction export for object-shaped RGB.
